# Lekce 08 - Vzor návrhu vícero agentů


## Nastavení


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity --quiet

import os
import asyncio

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Proč víceagentní systémy?

Reálné úkoly, jako je plánování cesty, zahrnují mnoho různých druhů odborností — logistiku, místní znalosti, rozpočtování a další. Jeden agent, který se snaží zvládnout všechno, se rychle stává neudržitelným.

Víceagentní systémy toto řeší pomocí **specializace**: každý agent se zaměřuje na jednu oblast odbornosti, čímž produkuje výsledky vyšší kvality než generalista. Zlepšují také **škálovatelnost** — můžete přidávat nové agenty (např. specialistu na lety, kritika restaurací) bez přepisování stávajícího pracovního postupu. Agenti se skládají dohromady prostřednictvím strukturovaného procesu, předávají si kontext jeden po druhém.


## Vytváření specializovaných agentů


In [ ]:
planner_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## Vytváření sekvenčního pracovního postupu

`WorkflowBuilder` vám umožňuje propojit agenty do směrovaného grafu. Zde vytvoříme jednoduchý dvoukrokový proces: **TravelPlanner** sestaví itinerář, poté ho **TravelConcierge** zkontroluje a vylepší.


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Přidání více agentů do pracovního postupu

Jednou z největších výhod vzoru s více agenty je, jak je snadné jej rozšířit. Níže přidáváme agenta **BudgetReviewer**, který kontroluje plán vůči rozpočtu cestovatele, označuje položky, které by mohly náklady překročit limit, a navrhuje úsporné alternativy. Pracovní postup nyní spouští tři agenty za sebou:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = await provider.create_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Shrnutí

V této lekci jste se naučili, jak:

1. **Vytvořit specializované agenty** — každý s jasně definovanou rolí (plánování, concierge, přezkum rozpočtu).
2. **Propojit agenty do sekvenčního pracovního postupu** pomocí `WorkflowBuilder` a `add_edge`.
3. **Streamovat výstup** z víceagentního pipeline, sledovat, který agent právě mluví.
4. **Rozšířit pracovní postup** přidáním nových agentů do řetězce bez úpravy již existujících.

Víceagentní designový vzor udržuje každého agenta jednoduchého a zároveň vytváří bohatší a důkladněji zkontrolované výsledky, než jakých by dosáhl jediný agent.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Prohlášení o vyloučení odpovědnosti**:
Tento dokument byl přeložen pomocí AI překladatelské služby [Co-op Translator](https://github.com/Azure/co-op-translator). Přestože usilujeme o přesnost, mějte prosím na paměti, že automatizované překlady mohou obsahovat chyby nebo nepřesnosti. Původní dokument v jeho mateřském jazyce by měl být považován za autoritativní zdroj. Pro kritické informace se doporučuje využít profesionální lidský překlad. Nejsme odpovědní za jakékoliv neporozumění nebo chybné interpretace vyplývající z použití tohoto překladu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
